# Dataset
- download
- preprocess

In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "xx"
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"] = "xx"

#!sed -i 's/\r$//' /content/drive/MyDrive/apai/fetch_src.sh  # Windows users
!bash /content/drive/MyDrive/apai/fetch_src.sh

#!sed -i 's/\r$//' /content/drive/MyDrive/apai/setup_colab.sh  # Windows users
!bash /content/drive/MyDrive/apai/setup_colab.sh

# imports
%run /content/src/imports.py
criterion = nn.CrossEntropyLoss()

In [ ]:
preprocess_dataset()

In [ ]:
class_to_idx_10 = {cls: i for i, cls in enumerate(SELECTED_CLASSES)}
print(class_to_idx_10)

TRAIN_DIR = os.path.join(OUTPUT_ROOT, "train")
VAL_DIR = os.path.join(OUTPUT_ROOT, "val")
TEST_DIR = os.path.join(OUTPUT_ROOT, "test")

train_dataset_10 = UCF101Clips(
    TRAIN_DIR,
    class_to_idx_10
)

val_dataset_10 = UCF101Clips(
    VAL_DIR,
    class_to_idx_10
)

test_dataset_10 = UCF101Clips(
    TEST_DIR,
    class_to_idx_10
)

train_loader_10 = DataLoader(train_dataset_10, batch_size=BATCH_SIZE, shuffle=True)
val_loader_10   = DataLoader(val_dataset_10, batch_size=BATCH_SIZE, shuffle=False)
test_loader_10  = DataLoader(test_dataset_10, batch_size=BATCH_SIZE, shuffle=False)

# MODEL
1. ViT b 16

In [ ]:
# model loading
vit_16 = vit_lstm(classes=len(SELECTED_CLASSES))

Model initialized!


In [ ]:
vit_16

vit_lstm(
  (vit): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
    

In [ ]:
# test before fine-tune
test_acc_vit_16, all_preds_vit_16, all_labels_vit_16 = test_model(vit_16, test_loader_10, device=device)

# Train Model
- Finetune

In [ ]:
results_vit_16 = train_model(
    vit_16,
    train_loader_10,
    val_loader_10,
    num_epochs=5
)

In [ ]:
plot_training_results(results_vit_16)

In [ ]:
# test after fine-tune
test_acc_vit_16, all_preds_vit_16, all_labels_vit_16 = test_model(vit_16, test_loader_10, device=device)

In [ ]:
plot_confusion_matrix(all_labels_results_vit_16, all_preds_results_vit_16, num_classes=10, classes_list=SELECTED_CLASSES)

In [ ]:
print_detailed_metrics(all_labels_results_vit_16, all_preds_results_resnet_vit_16, num_classes=10, classes_list=SELECTED_CLASSES)